# A/B Test Analysis: Cookie Cats
Statistical tests for the gate_30 vs gate_40 experiment.
Experiment design and decision rule: see README.
Data: cleaned in `01_cleaning.ipynb`.

## Setup

In [4]:
import pandas as pd
import numpy as np
from scipy.stats import norm

df_clean = pd.read_csv('../data/raw/cookie_cats_clean.csv')
print(df_clean.shape)

(90188, 5)


## Minimum detectable effect (MDE)

Before looking at results, I check how small a difference this test can reliably detect.
If the real difference is smaller than the MDE, the test will probably not find it.

For each metric, I use the overall retention (both groups together) as the baseline,
so I don't see the difference between groups yet.
Power = 80% (the test finds a real effect 8 times out of 10).
I use the smaller group size, to be conservative.

In [5]:
def mde(p, n, alpha, power=0.80):
    """Minimum detectable effect for a two-proportion test (absolute, as a share)."""
    z_alpha = norm.ppf(1 - alpha / 2)   # critical value for a two-sided test
    z_power = norm.ppf(power)           # 0.84 for 80% power
    return (z_alpha + z_power) * np.sqrt(2 * p * (1 - p) / n)

# smaller group size, to be conservative
n = df_clean['version'].value_counts().min()

### MDE for retention_7 (primary metric, α = 0.05)

In [6]:
# baseline: retention_7 for all players together (no group comparison yet)
p7 = df_clean['retention_7'].mean()
mde7 = mde(p7, n, alpha=0.05)

print(f'baseline retention_7: {p7:.2%}')
print(f'MDE: {mde7 * 100:.2f} percentage points ({mde7 / p7:.1%} of baseline)')

baseline retention_7: 18.61%
MDE: 0.73 percentage points (3.9% of baseline)


### Where 1.96 comes from

1.96 is the **critical value** for a two-sided test at α = 0.05.

If there is no real difference between the groups, the z-value is just random noise.
In that case, 95% of z-values fall between −1.96 and +1.96.

So:
- |z| ≤ 1.96 → this result can easily happen by chance → **not significant**
- |z| > 1.96 → this result happens by chance less than 5% of the time → **significant**

The 5% (α) is split into 2.5% on each side, because the test is two-sided
(gate_40 could be higher or lower than gate_30).

For the secondary metrics I use α = 0.025 (Bonferroni correction),
so the critical value is **2.24**.

z has no unit. It is not measured in percentage points.
The effect size (the difference in pp) and z are two different numbers.

### MDE for retention_1 (secondary metric, α = 0.025)

In [7]:
# baseline: retention_1 for all players together (no group comparison yet)
p1 = df_clean['retention_1'].mean()
mde1 = mde(p1, n, alpha=0.025)

print(f'baseline retention_1: {p1:.2%}')
print(f'MDE: {mde1 * 100:.2f} percentage points ({mde1 / p1:.1%} of baseline)')

baseline retention_1: 44.52%
MDE: 1.02 percentage points (2.3% of baseline)


## sum_gamerounds (secondary metric, α = 0.025)

Game rounds are heavily skewed (median 16, mean 51), so I use the Mann-Whitney U test.
It checks whether players in one group tend to play more rounds than in the other,
without being driven by a few very active players.

In [8]:
from scipy.stats import mannwhitneyu

g30 = df_clean.loc[df_clean['version'] == 'gate_30', 'sum_gamerounds']
g40 = df_clean.loc[df_clean['version'] == 'gate_40', 'sum_gamerounds']

print(f'median rounds: gate_30 = {g30.median()}, gate_40 = {g40.median()}')

result = mannwhitneyu(g40, g30, alternative='two-sided')
# chance that a random gate_40 player played more rounds than a random gate_30 player
prob_superiority = result.statistic / (len(g40) * len(g30))

print(f'p-value: {result.pvalue:.4f}')
print(f'probability of superiority (gate_40 > gate_30): {prob_superiority:.1%}')

median rounds: gate_30 = 17.0, gate_40 = 16.0
p-value: 0.0509
probability of superiority (gate_40 > gate_30): 49.6%


## Cross-check: z-tests in Python

I calculated the z-tests by hand in SQL (`sql/04_ztest_retention.sql`).
Here I run the same tests with statsmodels to check that the SQL is correct,
and to get the exact p-value and the confidence interval for the difference.

In [9]:
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

def ztest_retention(metric, alpha):
    # order: gate_40 first, so a negative difference means gate_40 is lower (same as in SQL)
    groups = df_clean.groupby('version')[metric]
    retained = [groups.sum()['gate_40'], groups.sum()['gate_30']]
    n = [groups.count()['gate_40'], groups.count()['gate_30']]

    z, p_value = proportions_ztest(retained, n)
    ci_low, ci_high = confint_proportions_2indep(retained[0], n[0], retained[1], n[1],
                                                 compare='diff', alpha=alpha)

    print(f'{metric} (α = {alpha})')
    print(f'  z = {z:.3f}, p-value = {p_value:.4f}')
    print(f'  {1 - alpha:.1%} CI for the difference: [{ci_low * 100:.2f}, {ci_high * 100:.2f}] percentage points')

ztest_retention('retention_7', alpha=0.05)
ztest_retention('retention_1', alpha=0.025)

retention_7 (α = 0.05)
  z = -3.157, p-value = 0.0016
  95.0% CI for the difference: [-1.33, -0.31] percentage points
retention_1 (α = 0.025)
  z = -1.787, p-value = 0.0739
  97.5% CI for the difference: [-1.33, 0.15] percentage points
